# Train YOLO26m - Object Detection

Train mô hình YOLO26m cho 7 lớp object theo format YOLO. Notebook không dùng test để train hoặc tạo pseudo-label.

In [1]:
# Nếu môi trường chưa có Ultralytics, chạy cell sau một lần:
# %pip install -U ultralytics

from pathlib import Path
import random
import shutil
import yaml
from ultralytics import YOLO

ROOT = Path.cwd()
DATA = ROOT / 'data'
if not DATA.exists():
    for parent in ROOT.parents:
        if (parent / 'data').exists():
            ROOT, DATA = parent, parent / 'data'
            break

TRAIN_SRC = DATA / 'train'
CLASSES = [x.strip() for x in (DATA / 'classes.txt').read_text(encoding='utf-8').splitlines() if x.strip()]
WORK = ROOT / 'yolo26m_work'
SPLIT_SEED = 42
VAL_RATIO = 0.2

print('classes:', CLASSES)
print('train images:', len(list((TRAIN_SRC / 'images').glob('*.png'))))

classes: ['bottle', 'cup', 'phone', 'book', 'mouse', 'box', 'toy_car']
train images: 6000


## Tạo train/validation split cố định

Split chỉ dùng dữ liệu train; test hoàn toàn không được đụng tới trong quá trình huấn luyện.

In [2]:
rng = random.Random(SPLIT_SEED)
images = sorted((TRAIN_SRC / 'images').glob('*.png'))
rng.shuffle(images)
n_val = int(len(images) * VAL_RATIO)
val_images, train_images = images[:n_val], images[n_val:]

for split in ('train', 'val'):
    (WORK / split / 'images').mkdir(parents=True, exist_ok=True)
    (WORK / split / 'labels').mkdir(parents=True, exist_ok=True)

for split, paths in [('train', train_images), ('val', val_images)]:
    for image_path in paths:
        label_path = TRAIN_SRC / 'labels' / f'{image_path.stem}.txt'
        shutil.copy2(image_path, WORK / split / 'images' / image_path.name)
        shutil.copy2(label_path, WORK / split / 'labels' / label_path.name)

data_yaml = WORK / 'data.yaml'
data_yaml.write_text(yaml.safe_dump({
    'path': str(WORK.resolve()),
    'train': 'train/images',
    'val': 'val/images',
    'names': {i: name for i, name in enumerate(CLASSES)},
}, sort_keys=False, allow_unicode=True), encoding='utf-8')
print(f'train={len(train_images)}, val={len(val_images)}')
print(data_yaml.read_text(encoding='utf-8'))

train=4800, val=1200
path: C:\Users\admin\OneDrive - The University of Technology\Desktop\edge_ai\yolo26m_work
train: train/images
val: val/images
names:
  0: bottle
  1: cup
  2: phone
  3: book
  4: mouse
  5: box
  6: toy_car



## Huấn luyện YOLO26m

Có thể điều chỉnh `batch` theo VRAM. Với RTX 3050 16GB, bắt đầu bằng batch 8 hoặc 16.

In [3]:
model = YOLO('yolo26m.pt')
results = model.train(
    data=str(data_yaml),
    imgsz=640,
    epochs=120,
    batch=8,
    device=0,
    workers=2,
    pretrained=True,
    optimizer='auto',
    cos_lr=True,
    patience=25,
    project=str(ROOT / 'runs'),
    name='yolo26m_objects',
    exist_ok=True,
    seed=SPLIT_SEED,
    deterministic=True,
    amp=True,
    plots=True,
)
RUN_DIR = ROOT / 'runs' / 'yolo26m_objects'
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
LAST_PT = RUN_DIR / 'weights' / 'last.pt'
assert BEST_PT.exists(), f'Missing best checkpoint: {BEST_PT}'
assert LAST_PT.exists(), f'Missing last checkpoint: {LAST_PT}'
print('best checkpoint:', BEST_PT)
print('last checkpoint:', LAST_PT)

New https://pypi.org/project/ultralytics/8.4.208 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.198  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=c:\Users\admin\OneDrive - The University of Technology\Desktop\edge_ai\yolo26m_work\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=120, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, l

KeyboardInterrupt: 

In [ ]:
RUN_DIR = ROOT / 'runs' / 'yolo26m_objects'
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
best = YOLO(str(BEST_PT))
metrics = best.val(data=str(data_yaml), imgsz=640, batch=8, device=0)
print('mAP50-95:', metrics.box.map)
print('mAP50:', metrics.box.map50)

Ultralytics 8.4.198  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO26m summary (fused): 130 layers, 20,354,849 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access  (ping: 0.0±0.0 ms, read: 911.4±391.8 MB/s, size: 84.2 KB)
val: Scanning C:\Users\admin\OneDrive - The University of Technology\Desktop\edge_ai\yolo26m_work\val\labels.cache... 1200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1200/1200  0.0s
